<a href="https://colab.research.google.com/github/Le2se0hy/XAI_An/blob/main/HPM_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 0. 라이브러리 설치
# ============================================================
!pip -q install openpyxl statsmodels scikit-learn

# ============================================================
# 1. 라이브러리 불러오기
# ============================================================
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.neighbors import BallTree
from google.colab import drive
from collections import OrderedDict

# ============================================================
# 2. Google Drive 연결
# ============================================================
drive.mount('/content/drive', force_remount=True)

# ============================================================
# 3. 파일 로드
# ============================================================
file_path = '/content/drive/MyDrive/!Seoul_Aprtment_FINAL.xlsx'
df_raw = pd.read_excel(file_path)

print("원본 데이터 shape:", df_raw.shape)
display(df_raw.head())
print("\n컬럼 목록:")
print(df_raw.columns.tolist())

# ============================================================
# 4. 숫자 정리 함수
# ============================================================
def clean_numeric_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", "", regex=False)
         .str.replace(" ", "", regex=False)
         .str.replace("\u00a0", "", regex=False)
         .replace({
             "-": np.nan,
             "": np.nan,
             "nan": np.nan,
             "None": np.nan,
             "null": np.nan,
             "NULL": np.nan
         }),
        errors="coerce"
    )

# ============================================================
# 5. 전처리 함수
# ============================================================
def prepare_seoul_df_final(df_input):
    df = df_input.copy()

    # -----------------------------
    # 숫자형 컬럼 강제 정리
    # -----------------------------
    numeric_candidates = [
        "Year_Sold", "Month_Sold",
        "Log_Price_per_m2",
        "Population",
        "Sex_ratio",
        "Pop. Density",
        "Median age",
        "Young Population",
        "Parking_per_Household",
        "Log_Dist_Water",
        "Log_Dist_Green",
        "Log_Dist_Subway",
        "max_floor",
        "num_of_people",
        "Bus_Stop",
        "High_School_Count",
        "Latitude",
        "Longitude",
        "Floor",
        "Size_m2",
        "Construction_Year",
        "heating"
    ]

    for col in numeric_candidates:
        if col in df.columns:
            df[col] = clean_numeric_series(df[col])

    # -----------------------------
    # heating dummy
    # 원래 heating이 숫자 1/0일 수도 있고 문자열일 수도 있어서 둘 다 대응
    # -----------------------------
    if "heating" in df.columns:
        raw_heat = df_input["heating"].astype(str).str.strip()
        df["heating_dummy"] = raw_heat.apply(
            lambda x: 1 if ("도시가스" in x or x == "1" or x == "1.0") else 0
        )
    else:
        df["heating_dummy"] = np.nan

    # -----------------------------
    # 연도/월
    # -----------------------------
    if "Year_Sold" not in df.columns:
        raise ValueError("Year_Sold 컬럼이 필요합니다.")
    if "Month_Sold" not in df.columns:
        raise ValueError("Month_Sold 컬럼이 필요합니다.")

    df["Analysis_Year"] = clean_numeric_series(df["Year_Sold"])
    df["Analysis_Month"] = clean_numeric_series(df["Month_Sold"])

    # -----------------------------
    # 계절 더미
    # -----------------------------
    df["Spring"] = df["Analysis_Month"].isin([3, 4, 5]).astype(int)
    df["Fall"]   = df["Analysis_Month"].isin([9, 10, 11]).astype(int)
    df["Winter"] = df["Analysis_Month"].isin([12, 1, 2]).astype(int)

    # -----------------------------
    # 논문표/비교표용 별칭 변수
    # 원 변수는 그대로 두고, 별칭만 추가
    # -----------------------------
    if "Size_m2" in df.columns:
        df["Size_Level"] = df["Size_m2"]

    if "num_of_people" in df.columns:
        df["Units_Level"] = df["num_of_people"]

    if "Pop. Density" in df.columns:
        df["Pop_Density_Level"] = df["Pop. Density"]


    # 비율 변수는 필요시 0~1로 맞춰서 별칭 생성
    ratio_map = {
        "Sex_ratio": "Sex_ratio_ratio",
        "Median age": "Medium_age_ratio"
    }

    for original, newcol in ratio_map.items():
        if original in df.columns:
            mx = df[original].dropna().max()
            if pd.notna(mx) and mx > 2:
                df[newcol] = df[original] / 100.0
            else:
                df[newcol] = df[original]

    return df

# ============================================================
# 6. Spatial Lag 계산 함수 (효율 개선 버전)
# ============================================================
def compute_WY_final(df_sub, y_col="Log_Price_per_m2", lat_col="Latitude", lon_col="Longitude", dist_band_km=1.0):
    """
    같은 좌표는 묶어서 평균 Y 사용
    1km 이내 이웃에 대해 inverse distance weight
    row-standardized weighted neighborhood mean 성격의 WY 생성
    """
    if len(df_sub) < 2:
        return np.zeros(len(df_sub))

    work = df_sub[[lat_col, lon_col, y_col]].copy().dropna()

    # 유일 좌표별 집계
    g = (
        work.groupby([lat_col, lon_col])[y_col]
        .agg(["sum", "count"])
        .reset_index()
    )

    lat_r = np.deg2rad(g[lat_col].values)
    lon_r = np.deg2rad(g[lon_col].values)
    coords = np.column_stack([lat_r, lon_r])

    tree = BallTree(coords, metric="haversine")
    rad_band = dist_band_km / 6371.0

    # 전체 query
    indices = tree.query_radius(coords, r=rad_band * 1.000001)

    g_sum = g["sum"].values
    g_count = g["count"].values
    WY_uni = np.zeros(len(g))

    for i, idx in enumerate(indices):
        idx = idx[idx != i]
        if idx.size == 0:
            continue

        # haversine distance
        dlat = (lat_r[idx] - lat_r[i]) / 2
        dlon = (lon_r[idx] - lon_r[i]) / 2
        a = np.sin(dlat)**2 + np.cos(lat_r[i]) * np.cos(lat_r[idx]) * np.sin(dlon)**2
        d = 6371.0 * 2.0 * np.arcsin(np.sqrt(a))

        mask = (d <= dist_band_km) & (d > 0)
        if not np.any(mask):
            continue

        invd = 1.0 / d[mask]

        # MATLAB 코드 취지를 살려서 "주변값 가중평균" 형태
        numerator = np.sum(invd * g_sum[idx[mask]])
        denominator = np.sum(invd * g_count[idx[mask]])

        if denominator > 0:
            WY_uni[i] = numerator / denominator

    g["_wy_val"] = WY_uni

    merged = df_sub.merge(
        g[[lat_col, lon_col, "_wy_val"]],
        on=[lat_col, lon_col],
        how="left"
    )

    return merged["_wy_val"].fillna(0.0).values

# ============================================================
# 7. 유의성 표시 함수
# ============================================================
def significance_marker(p):
    if pd.isna(p):
        return ""
    elif p < 0.01:
        return "‡"
    elif p < 0.05:
        return "†"
    elif p < 0.10:
        return "*"
    else:
        return ""

# ============================================================
# 8. 성능 요약 함수
# ============================================================
def model_summary_table(model, model_name, n_obs, best_rho=None):
    row = {
        "Model": model_name,
        "N": n_obs,
        "R2": model.rsquared,
        "Adj_R2": model.rsquared_adj,
        "RMSE": np.sqrt(model.mse_resid),
        "AIC": model.aic,
        "BIC": model.bic,
        "F_stat": model.fvalue if model.fvalue is not None else np.nan,
        "Prob_F": model.f_pvalue if model.f_pvalue is not None else np.nan,
        "Best_rho": best_rho if best_rho is not None else np.nan
    }
    return pd.DataFrame([row])

# ============================================================
# 9. 연도별 OLS / SLR 함수
# criterion: "AIC" 또는 "RMSE"
# ============================================================
def run_ols_slr_by_year(
    df_input,
    year_value,
    feature_cols,
    target_col="Log_Price_per_m2",
    lat_col="Latitude",
    lon_col="Longitude",
    dist_band_km=1.0,
    rho_grid=None,
    criterion="AIC"
):
    if rho_grid is None:
        rho_grid = np.round(np.arange(0.0, 0.95, 0.05), 2)

    df_year = df_input[df_input["Analysis_Year"] == year_value].copy()

    required_cols = feature_cols + [target_col, lat_col, lon_col]
    required_cols = [c for c in required_cols if c in df_year.columns]

    print(f"\n{'='*70}")
    print(f"{year_value}년 분석 시작")
    print(f"{'='*70}")
    print(f"[원본] {year_value}년 행 수:", len(df_year))

    print("\n[결측 상위 10개]")
    print(df_year[required_cols].isna().sum().sort_values(ascending=False).head(10))

    # 결측 제거
    df_year = df_year.dropna(subset=required_cols).copy().reset_index(drop=True)
    print(f"\n[최종 분석] {year_value}년 행 수:", len(df_year))

    if len(df_year) < 30:
        print(f"[경고] {year_value}년 데이터 부족 ({len(df_year)}행), 건너뜀")
        return None

    X = sm.add_constant(df_year[feature_cols], has_constant="add")
    y = df_year[target_col]

    # -----------------------
    # OLS
    # -----------------------
    ols_model = sm.OLS(y, X).fit()

    # -----------------------
    # Spatial Lag용 WY 계산
    # -----------------------
    df_year["WY"] = compute_WY_final(
        df_year,
        y_col=target_col,
        lat_col=lat_col,
        lon_col=lon_col,
        dist_band_km=dist_band_km
    )

    # -----------------------
    # rho 탐색
    # -----------------------
    best_rho = None
    best_model = None

    if criterion.upper() == "AIC":
        best_score = np.inf
    else:
        best_score = np.inf  # RMSE도 작을수록 좋음

    for rho in rho_grid:
        y_temp = y - rho * df_year["WY"]
        temp_model = sm.OLS(y_temp, X).fit()

        if criterion.upper() == "AIC":
            score = temp_model.aic
        else:
            score = np.sqrt(temp_model.mse_resid)

        if score < best_score:
            best_score = score
            best_rho = rho
            best_model = temp_model

    slr_model = best_model

    # -----------------------
    # 성능표
    # -----------------------
    perf_ols = model_summary_table(ols_model, "OLS", len(df_year), best_rho=None)
    perf_slr = model_summary_table(slr_model, "SLR", len(df_year), best_rho=best_rho)
    performance_table = pd.concat([perf_ols, perf_slr], ignore_index=True)

    # -----------------------
    # 계수 비교표
    # -----------------------
    coef_table = pd.DataFrame({
        "Variable": ols_model.params.index,
        "OLS_Coef": ols_model.params.values,
        "OLS_pvalue": ols_model.pvalues.values,
        "OLS_sig": [significance_marker(p) for p in ols_model.pvalues.values],
        "SLR_Coef": slr_model.params.values,
        "SLR_pvalue": slr_model.pvalues.values,
        "SLR_sig": [significance_marker(p) for p in slr_model.pvalues.values]
    })

    # -----------------------
    # 예측값표
    # -----------------------
    pred_table = pd.DataFrame({
        "Actual": y.values,
        "OLS_Pred": ols_model.predict(X),
        "SLR_Pred": slr_model.predict(X),
        "WY": df_year["WY"].values
    })

    print(f"\n[{year_value}] Best rho = {best_rho:.2f} ({criterion.upper()} 기준)")
    print(performance_table)

    return {
        "year": year_value,
        "data": df_year,
        "ols_model": ols_model,
        "slr_model": slr_model,
        "best_rho": best_rho,
        "performance_table": performance_table,
        "coef_table": coef_table,
        "pred_table": pred_table
    }

# ============================================================
# 10. 논문용 비교표 생성
# ============================================================
def build_comparison_table_thesis(res_dict, row_map):
    table = pd.DataFrame(index=row_map.keys(), columns=res_dict.keys())

    for col, res in res_dict.items():
        for rname, vname in row_map.items():

            if vname is None:
                table.loc[rname, col] = ""

            elif vname in ("F", "RMSE", "AdjR2"):
                pass

            elif vname in res.params.index:
                val = res.params[vname]
                p = res.pvalues[vname]
                sig = significance_marker(p)
                table.loc[rname, col] = f"{val:.4f}{sig}"

            else:
                table.loc[rname, col] = "–"

        table.loc["F-statistics", col] = f"{res.fvalue:,.0f}‡" if pd.notna(res.fvalue) else "–"
        table.loc["RMSE", col] = f"{np.sqrt(res.mse_resid):.4f}"
        table.loc["Adjusted R2", col] = f"{res.rsquared_adj:.4f}"

    return table

# ============================================================
# 11. 전체 데이터 준비
# ============================================================
df_all = prepare_seoul_df_final(df_raw)

print("\n[전처리 후 shape]")
print(df_all.shape)

# ============================================================
# 12. 사용할 feature 목록
# 원래 변수명 유지 버전
# ============================================================
features_main = [
    "Population",
    "Sex_ratio",
    "Pop. Density",
    "Median age",
    "Young Population",
    "Parking_per_Household",
    "Log_Dist_Water",
    "Log_Dist_Green",
    "Log_Dist_Subway",
    "max_floor",
    "heating_dummy",
    "num_of_people",
    "Bus_Stop",
    "High_School_Count",
    "Floor",
    "Size_m2",
    "Construction_Year",
    "Spring",
    "Fall",
    "Winter"
]

available_features = [c for c in features_main if c in df_all.columns]

print("\n사용 가능한 변수:")
for i, col in enumerate(available_features, 1):
    print(f"{i}. {col}")

if "Analysis_Year" not in df_all.columns:
    raise ValueError("Analysis_Year 생성 실패")

# ============================================================
# 13. 연도별 분석 실행
# ============================================================
target_years = sorted(df_all["Analysis_Year"].dropna().astype(int).unique().tolist())
print("\n분석 대상 연도:", target_years)

results = {}

for yr in target_years:
    res = run_ols_slr_by_year(
        df_all,
        year_value=yr,
        feature_cols=available_features,
        target_col="Log_Price_per_m2",
        lat_col="Latitude",
        lon_col="Longitude",
        dist_band_km=1.0,
        rho_grid=np.round(np.arange(0.0, 0.95, 0.05), 2),  # 필요시 바꿔도 됨
        criterion="AIC"   # "RMSE"로 바꿔도 됨
    )
    if res is not None:
        results[yr] = res

# ============================================================
# 14. 전체 성능 요약표
# ============================================================
all_summary_list = []

for yr, res in results.items():
    temp = res["performance_table"].copy()
    temp.insert(0, "Year", yr)
    all_summary_list.append(temp)

if len(all_summary_list) > 0:
    all_years_summary = pd.concat(all_summary_list, ignore_index=True)
else:
    all_years_summary = pd.DataFrame()

print("\n[전체 연도 요약표]")
display(all_years_summary)

# ============================================================
# 15. 논문형 비교표 만들기
# ============================================================
results_for_thesis = OrderedDict()

for yr, res in results.items():
    results_for_thesis[f"{yr}_OLS"] = res["ols_model"]
    results_for_thesis[f"{yr}_SLR"] = res["slr_model"]

row_map = OrderedDict([
    ("--- Property Characteristics ---", None),
    ("Size", "Size_m2"),
    ("Floor", "Floor"),
    ("Units", "num_of_people"),
    ("Parking", "Parking_per_Household"),
    ("Construction Year", "Construction_Year"),
    ("Max Floor", "max_floor"),
    ("Heating Dummy", "heating_dummy"),

    ("--- Accessibility & Environment ---", None),
    ("Dist. Subway (Log)", "Log_Dist_Subway"),
    ("Dist. Green (Log)", "Log_Dist_Green"),
    ("Dist. Water (Log)", "Log_Dist_Water"),
    ("Bus Stop", "Bus_Stop"),
    ("High School Count", "High_School_Count"),

    ("--- Demographics ---", None),
    ("Population", "Population"),
    ("Young Population", "Young Population"),
    ("Sex Ratio", "Sex_ratio"),
    ("Pop. Density", "Pop. Density"),
    ("Median Age", "Median age"),

    ("--- Seasonality ---", None),
    ("Spring", "Spring"),
    ("Fall", "Fall"),
    ("Winter", "Winter"),

    ("Constant", "const"),
    ("F-statistics", "F"),
    ("RMSE", "RMSE"),
    ("Adjusted R2", "AdjR2"),
])

if len(results_for_thesis) > 0:
    thesis_table = build_comparison_table_thesis(results_for_thesis, row_map)
    print("\n[논문용 비교표]")
    display(thesis_table)
else:
    thesis_table = pd.DataFrame()

# ============================================================
# 16. 엑셀 저장
# ============================================================
output_file = '/content/drive/MyDrive/Seoul_Apartment_Integrated_OLS_SLR_Results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # 전체 요약
    if not all_years_summary.empty:
        all_years_summary.to_excel(writer, sheet_name='All_Years_Summary', index=False)

    # 논문형 표
    if not thesis_table.empty:
        thesis_table.to_excel(writer, sheet_name='Thesis_Style_Table', index=True)

    # 연도별 결과
    for yr, res in results.items():
        res["performance_table"].to_excel(writer, sheet_name=f'{yr}_Performance', index=False)
        res["coef_table"].to_excel(writer, sheet_name=f'{yr}_Coef', index=False)
        res["pred_table"].to_excel(writer, sheet_name=f'{yr}_Pred', index=False)

print(f"\n저장 완료: {output_file}")

Mounted at /content/drive
원본 데이터 shape: (162375, 41)


,Address_Full,Gu_Name,Dong,Jibun,Apartment_Name,Price_String,Size_m2,Floor,Construction_Year,Year_Sold,...,Log_Dist_Water,Parking_per_Household,Young Population,Median age,Old Population,Pop. Density,Sex_ratio,Price_per_m2,Log_Price_per_m2,Population
0,서울특별시 종로구 숭인동 296-19,종로구,숭인동,296-19,삼전솔하임2차,10200,15.000,7,2012,2022,...,6.251133,0.214286,0.051709,0.765055,0.183237,26741.379325,0.980859,6.800000e+06,15.732433,15510
1,서울특별시 종로구 명륜2가 4,종로구,명륜2가,4,아남1,118000,84.900,18,1995,2022,...,6.769297,0.997706,0.071774,0.762895,0.165331,14591.964300,0.927700,1.389870e+07,16.447306,16343
2,서울특별시 종로구 연건동 195-10,종로구,연건동,195-10,이화에수풀,19500,16.980,7,2014,2022,...,6.528250,0.300000,0.052755,0.758793,0.188452,9258.974400,0.902500,1.148410e+07,16.256474,7222
3,서울특별시 종로구 내수동 72,종로구,내수동,72,경희궁의아침3단지,200000,150.480,3,2004,2022,...,6.984716,2.100000,0.095304,0.716547,0.188148,7600.813000,0.812200,1.329080e+07,16.402583,9349
4,서울특별시 종로구 효제동 65-2,종로구,효제동,1965-02-01 00:00:00,포레스트힐시티,19000,16.672,6,2017,2022,...,6.301153,0.611842,0.034897,0.757492,0.207610,8453.333300,1.146400,1.139635e+07,16.248804,5072



컬럼 목록:
['Address_Full', 'Gu_Name', 'Dong', 'Jibun', 'Apartment_Name', 'Price_String', 'Size_m2', 'Floor', 'Construction_Year', 'Year_Sold', 'Month_Sold', 'Day_Sold', 'Price_Won', 'Latitude', 'Longitude', 'High_School_Count', 'Bus_Stop', 'num_of_people', 'heating', 'parking', 'max_floor', 'Dist_Subway', 'Matched_Subway', 'Dist_Green', 'Matched_Green', 'Dist_Water', 'Matched_Water', 'Dist_CBD', 'Matched_CBD', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Parking_per_Household', 'Young Population', 'Median age', 'Old Population', 'Pop. Density', 'Sex_ratio', 'Price_per_m2', 'Log_Price_per_m2', 'Population']

[전처리 후 shape]
(162375, 52)

사용 가능한 변수:
1. Population
2. Sex_ratio
3. Pop. Density
4. Median age
5. Young Population
6. Parking_per_Household
7. Log_Dist_Water
8. Log_Dist_Green
9. Log_Dist_Subway
10. max_floor
11. heating_dummy
12. num_of_people
13. Bus_Stop
14. High_School_Count
15. Floor
16. Size_m2
17. Construction_Year
18. Spring
19. Fall
20. Winter

분석 대상 연도: [2022, 20

,Year,Model,N,R2,Adj_R2,RMSE,AIC,BIC,F_stat,Prob_F,Best_rho
0,2022,OLS,10579,0.449110,0.448066,0.335990,6966.420411,7119.019561,430.367426,0.0,NaN
1,2022,SLR,10579,0.276172,0.274800,0.291470,3958.878360,4111.477510,201.416474,0.0,0.6
2,2023,OLS,31071,0.430276,0.429909,0.326311,18603.562416,18778.787050,1172.502377,0.0,NaN
3,2023,SLR,31071,0.340990,0.340566,0.281692,9466.408426,9641.633060,803.306838,0.0,0.4
4,2024,OLS,50697,0.438944,0.438722,0.342527,35258.914193,35444.420256,1982.325272,0.0,NaN
5,2024,SLR,50697,0.296333,0.296055,0.218757,-10205.299275,-10019.793212,1067.049433,0.0,0.9
6,2025,OLS,70028,0.410498,0.410329,0.362424,56603.478966,56795.768625,2437.453945,0.0,NaN
7,2025,SLR,70028,0.283069,0.282864,0.237872,-2371.954474,-2179.664815,1382.058326,0.0,0.9



[논문용 비교표]


,2022_OLS,2022_SLR,2023_OLS,2023_SLR,2024_OLS,2024_SLR,2025_OLS,2025_SLR
--- Property Characteristics ---,,,,,,,,
Size,-0.0000,-0.0007‡,-0.0007‡,-0.0012‡,-0.0004‡,-0.0019‡,-0.0008‡,-0.0026‡
Floor,0.0018‡,0.0023‡,0.0027‡,0.0024‡,0.0023‡,0.0020‡,0.0017‡,0.0019‡
Units,0.0001‡,0.0001‡,0.0000‡,0.0000‡,0.0000‡,0.0000‡,0.0000‡,0.0000‡
Parking,0.0187‡,0.0186‡,0.0180‡,0.0162‡,0.0156‡,0.0131‡,0.0170‡,0.0128‡
Construction Year,-0.0020‡,0.0001,-0.0006‡,0.0009‡,0.0009‡,0.0035‡,0.0013‡,0.0030‡
Max Floor,0.0120‡,0.0090‡,0.0109‡,0.0093‡,0.0122‡,0.0079‡,0.0124‡,0.0088‡
Heating Dummy,-0.1812‡,-0.1159‡,-0.1880‡,-0.1314‡,-0.1796‡,-0.0708‡,-0.1625‡,-0.0634‡
--- Accessibility & Environment ---,,,,,,,,
Dist. Subway (Log),-0.0644‡,-0.0084,-0.1116‡,-0.0608‡,-0.1115‡,0.0020,-0.1131‡,0.0030*



저장 완료: /content/drive/MyDrive/Seoul_Apartment_Integrated_OLS_SLR_Results.xlsx
